# Claim Processing Results
Loads every claim from cache (or runs the agent if no cache exists) and renders a visual summary of documents, extracted fields, validation issues, decisions, and priority ranking.

In [6]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, display

sys.path.insert(0, str(Path().resolve().parent))
load_dotenv(dotenv_path=Path().resolve().parent / '.env')

from core.agent import ClaimAgent
from core.chatbot import Chatbot
from core.models import Claim

CLAIMS_DIR = Path().resolve().parent / 'claims'

In [ ]:
# ── Colour palette ──────────────────────────────────────────────────────────
STATUS_COLOR   = {'complete': '#27ae60', 'incomplete': '#e74c3c', 'needs_review': '#e67e22'}
CONF_COLOR     = {'high': '#27ae60',     'medium': '#e67e22',     'low': '#e74c3c'}
DOC_STATUS_COLOR = {'present': '#27ae60', 'missing': '#e74c3c', 'duplicate': '#e67e22'}
SEV_COLOR      = {'blocking': '#e74c3c', 'warning': '#f39c12'}
SEV_ICON       = {'blocking': '🔴', 'warning': '🟡'}

def _badge(text, color):
    return (f'<span style="background:{color};color:#fff;padding:2px 8px;'
            f'border-radius:4px;font-size:0.85em;font-weight:600">{text}</span>')

def _td(content, color=None, bold=False):
    style = f'color:{color};' if color else ''
    if bold: style += 'font-weight:600;'
    return f'<td style="padding:4px 10px;{style}">{content}</td>'

def _th(text):
    return (f'<th style="padding:4px 10px;text-align:left;background:#f0f0f0;'
            f'font-weight:600;border-bottom:2px solid #ddd">{text}</th>')

def _section(title):
    return f'<h4 style="margin:16px 0 6px;color:#444;border-bottom:1px solid #eee;padding-bottom:4px">{title}</h4>'

TABLE_STYLE = 'border-collapse:collapse;width:100%;font-size:0.9em;margin-bottom:8px'
ROW_STYLE   = 'border-bottom:1px solid #eee'


def render_claim(claim: Claim) -> str:
    status_color = STATUS_COLOR.get(claim.status, '#888')
    parts = []

    # ── Header ───────────────────────────────────────────────────────────────
    parts.append(
        f'<div style="border:2px solid {status_color};border-radius:6px;'
        f'padding:10px 16px;margin-bottom:12px">'
        f'<span style="font-size:1.2em;font-weight:700">{claim.claim_id}</span>&nbsp;&nbsp;'
        f'{_badge(claim.status.upper(), status_color)}&nbsp;&nbsp;'
        f'<span style="color:#888;font-size:0.85em">uploaded {claim.uploaded_at[:10]}</span>'
        f'&nbsp;&nbsp;<span style="color:#888;font-size:0.85em">{claim.reply_count} reply round(s)</span>'
        f'</div>'
    )

    # ── Documents ────────────────────────────────────────────────────────────
    parts.append(_section('Documents'))
    rows = ''.join(
        f'<tr style="{ROW_STYLE}">'
        + _td(r.file_name)
        + _td(r.doc_type)
        + _td(
            _badge(r.doc_status.upper(), DOC_STATUS_COLOR.get(r.doc_status, '#888'))
            + (f' <small style="color:#888">{r.duplicate_type}</small>' if r.duplicate_type else '')
        )
        + _td(
            r.parse_status if r.parse_status == 'complete'
            else f'<span style="color:#e74c3c">{r.parse_status}</span>'
        )
        + '</tr>'
        for r in claim.doc_table
    )
    parts.append(
        f'<table style="{TABLE_STYLE}"><thead><tr>'
        + _th('File') + _th('Type') + _th('Status') + _th('Parse')
        + f'</tr></thead><tbody>{rows}</tbody></table>'
    )

    # ── Extracted fields ─────────────────────────────────────────────────────
    parts.append(_section('Extracted Fields <small style="font-weight:normal;color:#888">(merged — highest confidence wins)</small>'))
    if claim.extracted_fields:
        rows = ''.join(
            f'<tr style="{ROW_STYLE}">'
            + _td(f'{f.field_name} <small style="color:#aaa">({f.field_role})</small>')
            + _td(f.unified_value or '<i style="color:#aaa">not found</i>')
            + _td(_badge(f.confidence, CONF_COLOR.get(f.confidence, '#888')))
            + _td('✓' if f.valid else '✗', color='#27ae60' if f.valid else '#e74c3c', bold=True)
            + '</tr>'
            for f in claim.extracted_fields.values()
        )
        parts.append(
            f'<table style="{TABLE_STYLE}"><thead><tr>'
            + _th('Field') + _th('Value') + _th('Confidence') + _th('Valid')
            + f'</tr></thead><tbody>{rows}</tbody></table>'
        )
    else:
        parts.append('<p style="color:#aaa;font-style:italic">No fields extracted.</p>')

    # ── Validation issues ────────────────────────────────────────────────────
    if claim.validation_issues:
        parts.append(_section('Validation Issues'))
        items = []
        for vi in claim.validation_issues:
            icon  = SEV_ICON.get(vi.severity, '⚪')
            color = SEV_COLOR.get(vi.severity, '#888')
            resolved_tag = ' <span style="color:#27ae60">[resolved]</span>' if vi.resolved else ''
            resubmit_tag = (f' → <span style="color:#e67e22">resubmit: {vi.resubmit_doc}</span>'
                            if vi.resubmit_doc else '')
            items.append(
                f'<li style="margin:4px 0">{icon} '
                f'<span style="color:{color};font-weight:600">[{vi.severity}]</span> '
                f'<b>{vi.field_name or vi.issue_type}</b>: '
                f'<span style="color:#555">{vi.description}</span>'
                f'{resubmit_tag}{resolved_tag}</li>'
            )
        parts.append('<ul style="margin:4px 0;padding-left:20px">' + ''.join(items) + '</ul>')

    # ── Decision / reason ────────────────────────────────────────────────────
    parts.append(_section('Decision'))
    missing   = [r.doc_type for r in claim.doc_table if r.doc_status == 'missing']
    failed    = [r.file_name for r in claim.doc_table if r.parse_status == 'parse_failed']
    blocking  = [vi for vi in claim.validation_issues if vi.severity == 'blocking' and not vi.resolved]
    warnings  = [vi for vi in claim.validation_issues if vi.severity == 'warning'  and not vi.resolved]

    reasons = []
    if missing:  reasons.append(f'Missing doc(s): <b>{", ".join(missing)}</b>')
    if failed:   reasons.append(f'Parse failed: <b>{", ".join(failed)}</b>')
    if blocking: reasons.append(f'{len(blocking)} blocking inconsistenc{"y" if len(blocking)==1 else "ies"}')
    if warnings: reasons.append(f'{len(warnings)} warning(s) — staff notified, no action needed')
    if not reasons and claim.status == 'complete':
        reasons.append('All required documents present and all fields valid.')

    decision_text = ' &nbsp;|&nbsp; '.join(reasons) if reasons else 'No issues.'
    parts.append(
        f'<p style="margin:4px 0">'
        f'Status set to {_badge(claim.status.upper(), status_color)} — {decision_text}</p>'
    )

    # ── Full conversation log (outbound + inbound) ───────────────────────────
    if claim.conversation_log:
        parts.append(_section(f'Conversation Log ({len(claim.conversation_log)} round(s))'))
        for cr in claim.conversation_log:
            if cr.direction == 'outbound':
                label = '→ Agent → Customer'
                border_color = '#0055cc'
                label_style = 'color:#0055cc;font-weight:600'
            else:
                label = '← Customer → Agent'
                border_color = '#27ae60'
                label_style = 'color:#27ae60;font-weight:600'
            escaped = cr.message.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
            parts.append(
                f'<div style="border-left:3px solid {border_color};padding-left:10px;margin-bottom:8px">'
                f'<small style="{label_style}">{label}</small>'
                f'<pre style="background:#f8f8f8;border:1px solid #ddd;border-radius:4px;'
                f'padding:8px;font-size:0.82em;white-space:pre-wrap;margin:4px 0">{escaped}</pre>'
                f'</div>'
            )

    return '<div style="font-family:sans-serif;margin-bottom:32px">' + ''.join(parts) + '</div>'

In [8]:
# Load claims — use cache where possible, run agent for missing caches
claim_dirs = sorted(p for p in CLAIMS_DIR.iterdir() if p.is_dir() and not p.name.startswith('.'))

claims = []
for claim_dir in claim_dirs:
    cache = claim_dir / '.cache' / 'claim_state.json'
    if cache.exists():
        claims.append(Claim.model_validate_json(cache.read_text()))
        print(f'  [cache]  {claim_dir.name}')
    else:
        print(f'  [run]    {claim_dir.name} — no cache, running agent...')
        agent = ClaimAgent(chatbot=Chatbot())
        claims.append(agent.process_claim(str(claim_dir), use_cache=False))

print(f'\nLoaded {len(claims)} claim(s).')

  [cache]  CLM-001
  [cache]  CLM-002
  [cache]  CLM-003
  [cache]  CLM-004
  [cache]  CLM-005

Loaded 5 claim(s).


In [9]:
# Render each claim
for claim in claims:
    display(HTML(render_claim(claim)))

File,Type,Status,Parse
adjuster_note.png,settlement_breakdown,PRESENT,complete
finance_agreement.png,finance_agreement,PRESENT,complete
police_report.pdf,police_report,PRESENT,complete
settlement_breakdown.pdf,settlement_breakdown,PRESENT,complete
Field,Value,Confidence,Valid
VIN (required),1HGCM82633A004352,high,✓
date_of_loss (required),2026-02-14,high,✓
insurance_payout (required),18750.00,high,✓
outstanding_loan_balance (optional),22340.55,high,✓


File,Type,Status,Parse
customer_reply.txt,customer_reply,PRESENT,complete
finance_agreement.pdf,finance_agreement,PRESENT,complete
settlement_breakdown.png,settlement_breakdown,PRESENT,complete
[missing] police_report,police_report,MISSING,unprocessed
Field,Value,Confidence,Valid
VIN (required),3FADP4BJ7EM207685,high,✓
date_of_loss (required),2026-01-28,medium,✓
insurance_payout (required),12400.00,medium,✓
outstanding_loan_balance (optional),15890.20,high,✓


File,Type,Status,Parse
finance_agreement.pdf,finance_agreement,PRESENT,complete
police_report.png,police_report,PRESENT,complete
settlement_breakdown.pdf,settlement_breakdown,PRESENT,complete
settlement_breakdown_v2.pdf,settlement_breakdown,PRESENT,complete
Field,Value,Confidence,Valid
VIN (required),2T1BURHE5JC034182,high,✓
date_of_loss (required),2026-03-05,high,✓
insurance_payout (required),24100.00,high,✓
outstanding_loan_balance (optional),27650.00,high,✓


File,Type,Status,Parse
finance_agreement.png,finance_agreement,PRESENT,complete
police_report.png,police_report,PRESENT,complete
settlement_breakdown.png,settlement_breakdown,PRESENT,complete
tow_receipt.png,unknown,PRESENT,complete
Field,Value,Confidence,Valid
VIN (required),2022TESLAMODEL3,medium,✗
date_of_loss (required),2025-12-19,medium,✓
insurance_payout (required),432.18,medium,✓
outstanding_loan_balance (optional),19200.00,medium,✓


File,Type,Status,Parse
customer_reply.txt,customer_reply,PRESENT,complete
police_report.png,police_report,PRESENT,complete
settlement_breakdown.pdf,settlement_breakdown,PRESENT,complete
[missing] finance_agreement,finance_agreement,MISSING,unprocessed
Field,Value,Confidence,Valid
VIN (required),1N4AL3AP8JC231504,high,✓
date_of_loss (required),2026-03-28,high,✓
insurance_payout (required),31500.00,high,✓
outstanding_loan_balance (optional),35120.75,high,✓


In [10]:
# Priority ranking summary
from core.agent import ClaimAgent
from core.chatbot import Chatbot

_agent = ClaimAgent(chatbot=Chatbot())
records = _agent.prioritize_claims(claims)

rows = ''.join(
    f'<tr style="{ROW_STYLE}">'
    + _td(f'<b>#{rec.priority_rank}</b>')
    + _td(rec.claim_id, bold=True)
    + _td(_badge(rec.status.upper(), STATUS_COLOR.get(rec.status, '#888')))
    + _td('⚡ EXPRESS' if rec.express else '', color='#e67e22', bold=True)
    + _td(rec.uploaded_at[:10])
    + _td(f'<span style="color:#555">{rec.reason}</span>')
    + '</tr>'
    for rec in records
)

html = (
    '<div style="font-family:sans-serif">'
    '<h3 style="border-bottom:2px solid #ddd;padding-bottom:6px">Priority Ranking</h3>'
    f'<table style="{TABLE_STYLE}"><thead><tr>'
    + _th('Rank') + _th('Claim') + _th('Status') + _th('Express') + _th('Uploaded') + _th('Reason')
    + f'</tr></thead><tbody>{rows}</tbody></table></div>'
)
display(HTML(html))

Loading weights: 100%|██████████| 713/713 [00:01<00:00, 491.47it/s] 


Rank,Claim,Status,Express,Uploaded,Reason
#1,CLM-001,COMPLETE,,2026-04-22,All required documents present and valid — ready to finalize. Oldest submission prioritized.
#2,CLM-003,COMPLETE,,2026-04-22,All required documents present and valid — ready to finalize. Oldest submission prioritized.
#3,CLM-004,INCOMPLETE,⚡ EXPRESS,2026-04-22,Incomplete with fewer than 2 unresolved issues — express routing for quick customer response.
#4,CLM-002,INCOMPLETE,,2026-04-22,Incomplete claim — customer notification required.
#5,CLM-005,INCOMPLETE,,2026-04-22,Incomplete claim — customer notification required.
